# Task 4 – Advanced Analytics & Statistical Modeling
**ApexPlanet Software Pvt Ltd – Data Analytics Internship**

**Objective:** Apply statistical analysis and basic predictive modeling.

**Sections:** 1. Statistical Analysis · 2. Time Series Analysis · 3. Customer Segmentation (K-Means) · 4. Predictive Modeling

> **Dataset note:** Set `DATA_PATH` below to your own CSV (e.g. Superstore). If the file is not found, the notebook generates a Superstore-style sample dataset so every cell still runs.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

## 0. Load Data

In [ ]:
import os
DATA_PATH = 'superstore.csv'   # <-- change to your dataset path

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, encoding='latin-1')
    df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]
    # Expected columns: Order_Date, Segment, Region, Category, Sales, Quantity, Discount, Profit, Customer_ID
else:
    # Superstore-style sample data (used only if no CSV is found)
    n = 3000
    dates = pd.to_datetime('2022-01-01') + pd.to_timedelta(np.random.randint(0, 3*365, n), unit='D')
    month_boost = 1 + 0.35*np.isin(dates.month, [11, 12])           # seasonality
    trend = 1 + (dates - dates.min()).days / 1500                    # upward trend
    qty = np.random.randint(1, 10, n)
    disc = np.random.choice([0, 0.1, 0.2, 0.3, 0.4], n, p=[.4, .2, .2, .1, .1])
    sales = np.random.gamma(2, 60, n) * qty * month_boost * trend * (1 - disc/2)
    profit = sales * (0.28 - 0.9*disc + np.random.normal(0, 0.05, n))
    df = pd.DataFrame({
        'Order_Date': dates,
        'Customer_ID': ['C' + str(i).zfill(3) for i in np.random.randint(1, 301, n)],
        'Segment': np.random.choice(['Consumer', 'Corporate', 'Home Office'], n, p=[.52, .30, .18]),
        'Region': np.random.choice(['East', 'West', 'Central', 'South'], n),
        'Category': np.random.choice(['Furniture', 'Office Supplies', 'Technology'], n, p=[.2, .6, .2]),
        'Sales': sales.round(2), 'Quantity': qty, 'Discount': disc, 'Profit': profit.round(2)})

df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(df.shape); df.head()

---
# 1. Statistical Analysis (Day 21–22)
## 1.1 Descriptive statistics

In [ ]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
desc = df[num_cols].agg(['mean', 'median', 'std', 'skew']).T
desc['mode'] = [df[c].mode().iloc[0] for c in num_cols]
desc = desc[['mean', 'median', 'mode', 'std', 'skew']].round(3)
desc

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['Sales'], kde=True, ax=ax[0]); ax[0].set_title(f"Sales (skew = {df['Sales'].skew():.2f})")
sns.histplot(df['Profit'], kde=True, ax=ax[1], color='green'); ax[1].set_title(f"Profit (skew = {df['Profit'].skew():.2f})")
plt.tight_layout(); plt.show()

**Interpretation:** Sales is right-skewed (mean > median) – a few very large orders pull the average up, so the **median** is a better 'typical order' measure.

## 1.2 Hypothesis test – t-test (two groups)
**H0:** mean Profit of Consumer = mean Profit of Corporate  
**H1:** the means differ (α = 0.05)

In [ ]:
a = df.loc[df.Segment == 'Consumer', 'Profit']
b = df.loc[df.Segment == 'Corporate', 'Profit']
t, p = stats.ttest_ind(a, b, equal_var=False)   # Welch's t-test
print(f"t = {t:.3f}, p-value = {p:.4f}")
print('Reject H0 – significant difference' if p < 0.05 else 'Fail to reject H0 – no significant difference')

## 1.3 Hypothesis test – Chi-square (categorical relationship)
**H0:** Segment and Category are independent  
**H1:** they are associated (α = 0.05)

In [ ]:
ct = pd.crosstab(df['Segment'], df['Category'])
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(ct); print(f"\nchi2 = {chi2:.3f}, dof = {dof}, p-value = {p:.4f}")
print('Reject H0 – associated' if p < 0.05 else 'Fail to reject H0 – independent')

## 1.4 Confidence interval (95%) for mean Sales

In [ ]:
s = df['Sales']
ci = stats.t.interval(0.95, len(s)-1, loc=s.mean(), scale=stats.sem(s))
print(f"Mean Sales = {s.mean():.2f}\n95% CI = ({ci[0]:.2f}, {ci[1]:.2f})")

## 1.5 Business implications
- **Skewed sales:** report the median alongside the mean; target high-value orders separately.
- **t-test:** if segments differ significantly in profit, allocate marketing effort to the more profitable segment.
- **Chi-square:** a significant association means product mix should be tailored per segment; otherwise one common strategy works.
- **Confidence interval:** with 95% confidence the true average order value lies in the range above – use it for realistic revenue forecasts.

---
# 2. Time Series Analysis (Day 23–24)
## 2.1 Convert to time series & resample

In [ ]:
ts = df.set_index('Order_Date').sort_index()['Sales']
daily, weekly, monthly = ts.resample('D').sum(), ts.resample('W').sum(), ts.resample('MS').sum()
fig, ax = plt.subplots(3, 1, figsize=(11, 8))
for a_, s_, t_ in zip(ax, [daily, weekly, monthly], ['Daily', 'Weekly', 'Monthly']):
    s_.plot(ax=a_); a_.set_title(f'{t_} Sales')
plt.tight_layout(); plt.show()

## 2.2 Decompose: trend, seasonality, residuals (additive)

In [ ]:
m = monthly.copy()
trend = m.rolling(12, center=True).mean()                       # trend = 12-month centred moving average
detr = m - trend
seasonal = detr.groupby(detr.index.month).transform('mean')      # average month effect
resid = m - trend - seasonal

fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for a_, s_, t_ in zip(ax, [m, trend, seasonal, resid], ['Observed', 'Trend', 'Seasonality', 'Residual']):
    s_.plot(ax=a_); a_.set_ylabel(t_)
plt.tight_layout(); plt.show()

## 2.3 Simple moving-average forecast

In [ ]:
window = 3
ma = monthly.rolling(window).mean()
forecast = monthly.tail(window).mean()          # next-month forecast = mean of last 3 months
next_month = monthly.index[-1] + pd.offsets.MonthBegin(1)

plt.figure(figsize=(11, 4))
monthly.plot(label='Actual'); ma.plot(label=f'{window}-month MA')
plt.scatter([next_month], [forecast], color='red', zorder=5, label=f'Forecast: {forecast:,.0f}')
plt.legend(); plt.title('Monthly Sales & Moving Average Forecast'); plt.show()

valid = monthly.iloc[window:]; pred = monthly.rolling(window).mean().shift(1).iloc[window:]
mape = (abs(valid - pred) / valid).mean() * 100
print(f'Next-month forecast: {forecast:,.2f}  |  Backtest MAPE: {mape:.1f}%')

**Insight:** Sales show an upward trend with a clear year-end seasonal peak – plan inventory and promotions ahead of Q4.

---
# 3. Customer Segmentation – K-Means (Day 23–24)
## 3.1 Build customer features & scale

In [ ]:
cust = df.groupby('Customer_ID').agg(
    Total_Sales=('Sales', 'sum'), Total_Profit=('Profit', 'sum'),
    Orders=('Sales', 'count'), Avg_Discount=('Discount', 'mean'),
    Avg_Quantity=('Quantity', 'mean')).reset_index()
features = ['Total_Sales', 'Total_Profit', 'Orders', 'Avg_Discount', 'Avg_Quantity']
X = StandardScaler().fit_transform(cust[features])
cust.head()

## 3.2 Elbow method – optimal K

In [ ]:
inertia, K = [], range(1, 10)
for k in K:
    inertia.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_)
plt.plot(K, inertia, 'o-'); plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow Method'); plt.show()

## 3.3 Apply K-Means (K = 4) & visualise with PCA

In [ ]:
K_OPT = 4   # choose from the elbow plot
km = KMeans(n_clusters=K_OPT, n_init=10, random_state=42)
cust['Cluster'] = km.fit_predict(X)

pca = PCA(n_components=2)
pc = pca.fit_transform(X)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=pc[:, 0], y=pc[:, 1], hue=cust['Cluster'], palette='Set1', s=60)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})'); plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
plt.title('Customer Segments (PCA 2D)'); plt.show()

## 3.4 Cluster profiling & recommendations

In [ ]:
profile = cust.groupby('Cluster')[features].mean().round(2)
profile['Customers'] = cust['Cluster'].value_counts().sort_index()
profile

| Segment type | Typical profile | Recommended action |
|---|---|---|
| **VIP / Champions** | High sales, high profit, many orders | Loyalty rewards, early access, dedicated support |
| **Loyal regulars** | Many orders, medium value | Cross-sell and bundle offers |
| **Discount seekers** | High avg discount, low profit | Cap discounts, push higher-margin products |
| **Low-value / at-risk** | Few orders, low sales | Re-engagement emails, win-back campaigns |

*(Match each cluster number to a row using the profile table above.)*

---
# 4. Predictive Modeling (Day 25–26)
## 4.1 Target & features
**Target:** `Profit` (numeric → Linear Regression)

In [ ]:
model_df = df[['Sales', 'Quantity', 'Discount', 'Segment', 'Region', 'Category', 'Profit']]
X_m = pd.get_dummies(model_df.drop(columns='Profit'), drop_first=True)
y = model_df['Profit']
X_train, X_test, y_train, y_test = train_test_split(X_m, y, test_size=0.2, random_state=42)   # 80/20
print(X_train.shape, X_test.shape)

## 4.2 Train & evaluate

In [ ]:
scaler = StandardScaler()
Xtr, Xte = scaler.fit_transform(X_train), scaler.transform(X_test)
lr = LinearRegression().fit(Xtr, y_train)
pred = lr.predict(Xte)

print(f'R²   : {r2_score(y_test, pred):.3f}')
print(f'MAE  : {mean_absolute_error(y_test, pred):.2f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.2f}')

plt.scatter(y_test, pred, alpha=.4); plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Actual Profit'); plt.ylabel('Predicted Profit'); plt.title('Actual vs Predicted'); plt.show()

## 4.3 Top 3 important features

In [ ]:
imp = pd.Series(np.abs(lr.coef_), index=X_m.columns).sort_values(ascending=False)
print('Top 3 features:'); print(imp.head(3))
imp.head(8).sort_values().plot(kind='barh', title='Feature importance (|standardised coefficient|)'); plt.show()

---
# Conclusion
- Descriptive stats showed right-skewed sales; hypothesis tests and a 95% CI quantified differences between segments.
- Time-series analysis revealed trend + seasonality and produced a moving-average forecast.
- K-Means (K = 4) segmented customers into actionable groups.
- A Linear Regression model predicts profit; the top 3 drivers guide pricing and discount strategy.

#DataScience #MachineLearning #ApexPlanet